# Exercise 5 – LU Decomposition (Task 1)

In this notebook you implement the LU decomposition yourself – first without, then with pivoting.

**Rules for this part: no AI tools.** Write the code yourself and make sure you understand every line.
(In Task 2 on the exercise sheet you may use AI.)

**How it works:** Fill in the gaps marked `# YOUR CODE HERE` and delete the line `raise NotImplementedError`.
After each gap a check cell prints ✓ (correct) or ✗ (wrong). Run the cells from top to bottom (*Shift+Enter*).

> **Save your work!** This Binder session is temporary and is deleted after a few minutes of inactivity.
> Use *File → Download* now and then. You need your functions again for Task 2.

## Recap

The LU decomposition writes a square matrix as a product $A = LU$ of a lower triangular matrix $L$
(with ones on the diagonal) and an upper triangular matrix $U$. Then $A\mathbf x = \mathbf b$ is solved in two steps:

1. $L\mathbf y = \mathbf b$ by **forward substitution**: $\quad y_i = \Big(b_i - \sum_{j<i} l_{ij}\, y_j\Big) / l_{ii}, \quad i = 1, 2, \dots, n$
2. $U\mathbf x = \mathbf y$ by **back substitution**: $\quad x_i = \Big(y_i - \sum_{j>i} u_{ij}\, x_j\Big) / u_{ii}, \quad i = n, n-1, \dots, 1$

Note: Python indices start at 0.

In [ ]:
import numpy as np
import scipy.linalg

np.set_printoptions(precision=4, suppress=True)


def check(label, ok):
    print(("✓  " if ok else "✗  ") + label)

## 1  Triangular systems

Complete `forward_subst` and `back_subst`.
Tip: `L[i, :i] @ y[:i]` computes the sum $\sum_{j<i} l_{ij}\, y_j$.

In [ ]:
def forward_subst(L, b):
    """Solve L y = b for a lower triangular matrix L."""
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        # YOUR CODE HERE
        raise NotImplementedError
    return y


def back_subst(U, y):
    """Solve U x = y for an upper triangular matrix U."""
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):          # i = n-1, n-2, ..., 0
        # YOUR CODE HERE
        raise NotImplementedError
    return x

In [ ]:
# check
rng = np.random.default_rng(1)
T = np.tril(rng.uniform(-1, 1, (6, 6))) + 2 * np.eye(6)     # lower triangular test matrix
b = rng.uniform(-1, 1, 6)
check("forward_subst", np.allclose(forward_subst(T, b), scipy.linalg.solve_triangular(T, b, lower=True)))
check("back_subst",    np.allclose(back_subst(T.T, b),  scipy.linalg.solve_triangular(T.T, b, lower=False)))

## 2  LU decomposition without pivoting

Gaussian elimination: for every pivot column $k$ and every row $i > k$ below it

$$ l_{ik} = \frac{u_{ik}}{u_{kk}}, \qquad \text{row}_i \leftarrow \text{row}_i - l_{ik}\cdot \text{row}_k . $$

The factors $l_{ik}$ form $L$, the remaining upper triangle is $U$ (see the lecture example below).

Complete the body of the `i` loop. **Use explicit loops** (also over the columns `j`, no slicing) –
in Task 2 you will time exactly this version.

In [ ]:
def lu_nopivot(A):
    """LU decomposition A = L U without pivoting."""
    n = A.shape[0]
    U = np.array(A, dtype=float)     # a float copy of A, it becomes U  (why float?)
    L = np.eye(n)
    for k in range(n - 1):           # pivot column k
        for i in range(k + 1, n):    # rows below the pivot
            # YOUR CODE HERE
            raise NotImplementedError
    return L, U

In [ ]:
# check 1: example from the lecture
A1 = np.array([[1, 2, 3],
               [1, 1, 1],
               [3, 3, 1]])
L, U = lu_nopivot(A1)
print("L =\n", L, "\nU =\n", U)
check("lecture example: L", np.allclose(L, [[1, 0, 0], [1, 1, 0], [3, 3, 1]]))
check("lecture example: U", np.allclose(U, [[1, 2, 3], [0, -1, -2], [0, 0, -2]]))

# check 2: random diagonally dominant matrix
A2 = rng.uniform(-1, 1, (6, 6)) + 6 * np.eye(6)
L, U = lu_nopivot(A2)
check("L is lower triangular with unit diagonal", np.allclose(L, np.tril(L)) and np.allclose(np.diag(L), 1))
check("U is upper triangular", np.allclose(U, np.triu(U)))
check("L U = A", np.allclose(L @ U, A2))

Now combine the decomposition with the two substitutions (one or two lines).

In [ ]:
def lu_solve_nopivot(L, U, b):
    """Solve L U x = b."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check
L, U = lu_nopivot(A1)
x = lu_solve_nopivot(L, U, np.array([2, 2, 0]))
print("x =", x)
check("lecture example: x = (5, -6, 3)", np.allclose(x, [5, -6, 3]))

A3 = np.array([[ 7,  3, -1,  2],
               [ 3,  8,  1, -4],
               [-1,  1,  4, -1],
               [ 2, -4, -1,  6]])
b3 = np.array([1, 2, 3, 4])
L, U = lu_nopivot(A3)
x = lu_solve_nopivot(L, U, b3)
print("x =", x)
check("4x4 system, compared with np.linalg.solve", np.allclose(x, np.linalg.solve(A3, b3)))

## 3  When does it fail?

**Predict first, then run the cells.** Both matrices are invertible, so both systems have a unique solution.

**(a)** The anti-diagonal matrix
$$ J = \begin{bmatrix} 0&0&0&1\\0&0&1&0\\0&1&0&0\\1&0&0&0 \end{bmatrix} $$

In [ ]:
J = np.fliplr(np.eye(4))
L, U = lu_nopivot(J)
print("L =\n", L, "\nU =\n", U)

**(b)** The system
$$ \begin{bmatrix} \varepsilon & 1\\ 1 & 1 \end{bmatrix} \mathbf x = \begin{bmatrix} 1\\ 2 \end{bmatrix}
\qquad \text{with } \varepsilon = 10^{-20}, \qquad \text{exact solution } \mathbf x = \Big(\tfrac{1}{1-\varepsilon},\ \tfrac{1-2\varepsilon}{1-\varepsilon}\Big) \approx (1, 1). $$

In [ ]:
eps = 1e-20
A_eps = np.array([[eps, 1],
                  [1,   1]])
b_eps = np.array([1, 2])
L, U = lu_nopivot(A_eps)
print("L =\n", L, "\nU =\n", U)
print("x =", lu_solve_nopivot(L, U, b_eps), "  (exact: approx. [1, 1])")

**Questions** (discuss with your group):
- What goes wrong in (a)? Which line of your code causes it?
- In (b) nothing crashes, but the result is wrong. Compute $u_{22}$ by hand and compare with the printed `U`.
  Which information is lost? (Try `1 - 1e20` in Python.)
- Would it help to swap the two rows of the system first?

## 4  LU decomposition with partial pivoting

Before eliminating column $k$, look for the row $m \ge k$ with the **largest** $|u_{mk}|$ and swap rows $k$ and $m$.
Then the pivot is never zero (for invertible $A$) and $|l_{ik}| \le 1$.

The swaps are collected in a permutation vector `p`: row `i` of $PA$ is row `p[i]` of $A$, i.e. `P @ A == A[p]`.
The result is
$$ PA = LU . $$

In step $k$ the rows $k$ and $m$ have to be swapped in
- `U` (the whole row),
- `L` – but only in the columns `:k` that are already computed,
- `p`.

> **Careful:** `U[k], U[m] = U[m], U[k]` does **not** swap two rows of a NumPy array (try it with a small matrix – why?).
> Use fancy indexing instead: `U[[k, m]] = U[[m, k]]`.

Tip: `np.argmax(np.abs(U[k:, k]))` gives the position of the largest entry *relative to row k*.

In [ ]:
def lu_pivot(A):
    """LU decomposition with partial pivoting: P A = L U, P = I[p]. Returns p, L, U."""
    n = A.shape[0]
    U = np.array(A, dtype=float)
    L = np.eye(n)
    p = np.arange(n)
    for k in range(n - 1):
        # 1. find the pivot row m >= k   2. swap rows k and m in U, L (columns :k) and p
        # YOUR CODE HERE
        raise NotImplementedError

        # 3. eliminate below the pivot (as in lu_nopivot)
        # YOUR CODE HERE
        raise NotImplementedError
    return p, L, U


def lu_solve_pivot(p, L, U, b):
    """Solve A x = b with P A = L U."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# check 1: random general matrix
B = rng.uniform(-1, 1, (8, 8))
p, L, U = lu_pivot(B)
P = np.eye(8)[p]
check("P A = L U", np.allclose(P @ B, L @ U))
check("|l_ik| <= 1", np.all(np.abs(L) <= 1 + 1e-14))

# check 2: compare with scipy. Careful, scipy uses the convention A = P_s L U, i.e. P_s = P^T
P_s, L_s, U_s = scipy.linalg.lu(B)
check("same P, L, U as scipy.linalg.lu", np.allclose(P_s.T, P) and np.allclose(L_s, L) and np.allclose(U_s, U))

# check 3: the two problem cases from section 3
p, L, U = lu_pivot(J)
x = lu_solve_pivot(p, L, U, np.array([1, 2, 3, 4]))
check("anti-diagonal matrix: x = (4, 3, 2, 1)", np.allclose(x, [4, 3, 2, 1]))

p, L, U = lu_pivot(A_eps)
x = lu_solve_pivot(p, L, U, b_eps)
print("x =", x)
check("epsilon system: x = (1, 1)", np.allclose(x, [1, 1], rtol=1e-14))

**Questions:**
- Why does `forward_subst` get `b[p]` and not `b`?
- `scipy.linalg.lu` returns `P_s, L, U` with $A = P_s L U$. How is $P_s$ related to our $P$?
- Can pivoting still fail? For which matrices?

## 5  Save your functions for Task 2

The next cell writes your functions into the file `my_lu.py`.
Download it (file browser on the left → right click → *Download*), together with this notebook.
In Task 2 you can then use them with `from my_lu import *`.

In [ ]:
import inspect

functions = [forward_subst, back_subst, lu_nopivot, lu_solve_nopivot, lu_pivot, lu_solve_pivot]
with open("my_lu.py", "w") as f:
    f.write('"""LU decomposition, exercise 5, task 1."""\nimport numpy as np\n\n\n')
    f.write("\n\n".join(inspect.getsource(fn) for fn in functions))
print("written: my_lu.py")